# E1 on a GPU: Whisper-small, large-v3-turbo and large-v3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivannamor/multilingual-asr-lab/blob/main/notebooks/e1_large_colab.ipynb)

Transcribes the full FLEURS test sets (Italian, German, English) with three Whisper models in
float16 on a GPU, for experiment E1 of
[RESULTS.md](https://github.com/ivannamor/multilingual-asr-lab/blob/main/RESULTS.md). Whisper-small
is the *bridge* run: compared with the committed Whisper-small runs (int8 on a CPU), it measures
how much the change of hardware and precision alone moves the results.

This notebook only transcribes. Scores, comparisons and tables are computed from the saved
predictions by `experiments/E1/run.sh`, like every other result.

1. On Google Colab: *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. It takes
   about one and a half hours; keep the tab open.
2. At the end the browser downloads `e1_gpu_runs.zip` (if it does not, download it from the *Files*
   panel). It contains only `predictions.jsonl` and `run.json` of each run, no audio.
3. In a local clone: `unzip e1_gpu_runs.zip` from the repository root, then
   `bash experiments/E1/run.sh`.

If the session is interrupted, run all cells again: completed runs are skipped and an interrupted
run resumes where it stopped, as long as the Colab machine is the same.

## 1. Clone and install

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# Work inside a clone of the repository, so that every run.json records the git commit and the
# run directories end up where experiments/E1/run.sh expects them.
if Path("../pyproject.toml").exists():  # started from notebooks/ in a local clone
    os.chdir("..")
elif not Path("pyproject.toml").exists():  # on Colab
    if not Path("multilingual-asr-lab").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ivannamor/multilingual-asr-lab"],
            check=True,
        )
    os.chdir("multilingual-asr-lab")
if importlib.util.find_spec("asr_lab") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "."], check=True)
print(Path.cwd())

## 2. Settings

The models are transcribed in this order; the larger the model, the longer it takes (on a T4,
large-v3 is the slowest, about 20 to 30 minutes per language).

In [ ]:
LANGS = ["it", "de", "en"]
MODELS = ["small", "large-v3-turbo", "large-v3"]  # small: bridge run to the CPU int8 runs
BEAM_SIZE = 5  # as in every E1 run

### GPU setup

Same setup as the quickstart notebook: install the CUDA 12 libraries that CTranslate2 is built for
and load them explicitly. Unlike the quickstart, this notebook needs the GPU and stops if it is not
usable.

In [ ]:
import ctypes

import ctranslate2
import numpy as np

if ctranslate2.get_cuda_device_count() == 0:
    raise RuntimeError("No GPU: choose Runtime -> Change runtime type -> T4 GPU")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"],
    check=True,
)

In [ ]:
import nvidia.cublas
import nvidia.cudnn
from faster_whisper import WhisperModel

pending = [
    path
    for package in (nvidia.cublas, nvidia.cudnn)
    for path in sorted(Path(next(iter(package.__path__))).glob("lib/*.so*"))
]
for _ in range(5):  # the libraries depend on each other: retry the ones that failed
    failed = []
    for path in pending:
        try:
            ctypes.CDLL(str(path), mode=ctypes.RTLD_GLOBAL)
        except OSError:
            failed.append(path)
    pending = failed
    if not pending:
        break

check = WhisperModel("tiny", device="cuda", compute_type="float16")
list(check.transcribe(np.zeros(16_000, dtype=np.float32), language="it")[0])
print("GPU ready")

## 3. Data

The full test split of each language, built exactly as on a local machine, so that the manifest
has the same SHA-256 and the predictions can be scored locally.

In [ ]:
from asr_lab.data.ingest import ingest_fleurs
from asr_lab.provenance import file_sha256

manifests = {}
for lang in LANGS:
    manifest = Path(f"data/fleurs_{lang}/test.jsonl")
    if not manifest.exists():
        ingest_fleurs(lang, "test", manifest.parent)
    manifests[lang] = manifest
    print(f"{manifest}  sha256 {file_sha256(manifest)}")

## 4. Transcribe

In [ ]:
import gc

from asr_lab.inference.faster_whisper import FasterWhisperModel
from asr_lab.inference.run import transcribe_manifest

for size in MODELS:
    model = FasterWhisperModel(size, device="cuda", beam_size=BEAM_SIZE)
    for lang, manifest in manifests.items():
        run_dir = Path(f"experiments/runs/whisper-{size}-{model.compute_type}-fleurs-{lang}")
        print(run_dir)
        transcribe_manifest(model, manifest, run_dir)
    del model
    gc.collect()  # free the GPU memory before loading the next model

## 5. Download the predictions

In [ ]:
import json
import zipfile

archive = Path("e1_gpu_runs.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
    for run_dir in sorted(Path("experiments/runs").glob("whisper-*-float16-fleurs-*")):
        run = json.loads((run_dir / "run.json").read_text(encoding="utf-8"))
        print(f"{run_dir.name:45} RTF {run['rtf'] or float('nan'):.3f}")
        for name in ("predictions.jsonl", "run.json"):
            zipped.write(run_dir / name, (run_dir / name).as_posix())
print(f"{archive.resolve()}: {archive.stat().st_size / 1e6:.1f} MB")

try:
    from google.colab import files

    files.download(str(archive))
except ImportError:  # not on Colab
    pass